# 🗂️ Notebook 2: Discord — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/discord
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Core entities

| Table | Columns | Notes |
|---|---|---|
| `users` | id, name, avatar, presence | |
| `guilds` | id, name, owner_id | a "server" in Discord |
| `channels` | id, guild_id, type (text/voice), name | |
| `memberships` | user_id, guild_id, roles[] | join table |
| `messages` | id(time-ordered), channel_id, author_id, content, ts | **partition by channel_id** |

### Why partition messages by channel?
Reads are almost always "last N messages in this channel". A sequential scan of one partition
is O(N) with no joins. Cassandra and ScyllaDB are common picks — wide-column, per-channel partition.


## Key APIs & events

```http
# HTTP control plane
POST /channels/{id}/messages      { content }      → 201 { id }
GET  /channels/{id}/messages?before={id}&limit=50
POST /guilds                      create guild
POST /guilds/{id}/members         join

# WebSocket gateway (JSON events, like Discord's real protocol)
C→S:  { "op": "identify", "token": "..." }
S→C:  { "op": "hello",    "heartbeat_ms": 30000 }
S→C:  { "op": "ready",    "user": {...}, "guilds": [...] }
S→C:  { "op": "MESSAGE_CREATE", "d": { "channel_id":..., "content":... } }
S→C:  { "op": "PRESENCE_UPDATE", "d": { "user_id":..., "status":"online" } }
```

### Why a heartbeat?
TCP may keep a connection "open" long after the network is dead. App-level heartbeat
(every ~30s) proves liveness. Miss 2 heartbeats → assume dead, close socket, reconnect.


In [ ]:
from pydantic import BaseModel
from datetime import datetime
from typing import Literal, Optional

class Message(BaseModel):
    id: int
    channel_id: int
    author_id: int
    content: str
    ts: datetime

class GatewayEvent(BaseModel):
    op: Literal["hello","ready","MESSAGE_CREATE","PRESENCE_UPDATE","heartbeat"]
    d: Optional[dict] = None

e = GatewayEvent(op="MESSAGE_CREATE",
                 d=Message(id=1, channel_id=99, author_id=42,
                           content="hi", ts=datetime.utcnow()).model_dump(mode="json"))
print(e.model_dump_json(indent=2))
